# FlashAttentionCUDA — Nível 1: kernel com tiling + softmax online

Checklist deste notebook (Nível 1 do plano de desenvolvimento, semanas 2-3):

1. Confirmar GPU disponível (T4, tier grátis do Colab).
2. Clonar o repositório e compilar a extensão `flash_attention_tiled` (CUDA, tiling + softmax online).
3. Rodar o harness de corretude (kernel tiled vs. referência PyTorch *e* vs. kernel ingênuo).
4. Rodar o benchmark mínimo (tempo de execução + memória de pico, ingênuo vs. tiled vs. PyTorch nativo, variando N).

Pré-requisito: o notebook `00_setup_and_naive.ipynb` já deve ter sido executado com sucesso ao menos uma vez (confirma GPU, extensão ingênua e harness de corretude do Nível 0).

## 1. GPU disponível

In [ ]:
!nvidia-smi

## 2. Clonar o repositório e compilar

Se o repositório `IagoNat/HPC` for privado, troque a URL abaixo por uma
com um personal access token: `https://<token>@github.com/IagoNat/HPC.git`.

In [ ]:
!git clone https://github.com/IagoNat/HPC.git
%cd HPC/FlashAttentionCUDA

In [ ]:
# compila a extensão tiled (primeira chamada faz o build e já roda um teste mínimo)
import torch
from python.flash_attention_tiled import flash_attention_tiled

Q = torch.randn(1, 1, 8, 16, device="cuda", dtype=torch.float32)
K = torch.randn(1, 1, 8, 16, device="cuda", dtype=torch.float32)
V = torch.randn(1, 1, 8, 16, device="cuda", dtype=torch.float32)
out = flash_attention_tiled(Q, K, V)
print("compilou e rodou - shape de saída:", out.shape)

## 3. Harness de corretude

Compara o kernel tiled contra (a) a referência PyTorch e (b) o kernel ingênuo do Nível 0, por tolerância numérica (nunca por igualdade bit a bit). Inclui casos com N menor, igual e maior que o tile size (`BLOCK_N = 32`), para exercitar o caso de um único tile e o de múltiplos tiles (inclusive com último tile parcial).

In [ ]:
!python tests/test_tiled_correctness.py

## 4. Benchmark mínimo

Tempo de execução (eventos CUDA) e memória de pico (`torch.cuda.max_memory_allocated`), em função de N, comparando as três versões da Seção 1 da proposta. Ajuste `SEQ_LENS` em `python/benchmark.py` para incluir sequências maiores quando for montar o benchmark completo do Nível 2.

In [ ]:
!python python/benchmark.py

## Status do Nível 1

Preencher após rodar este notebook:

- [ ] Extensão `flash_attention_tiled` compila e roda na T4.
- [ ] Harness de corretude passa em todos os casos (vs. referência e vs. kernel ingênuo).
- [ ] Benchmark mínimo roda sem erro para N em `python/benchmark.py`.
- [ ] Kernel tiled é mais rápido e/ou usa menos memória de pico que o ingênuo nos N maiores testados (esperado; se não for o caso, registrar como achado para o relatório - pode indicar que o tile size ainda não está bem ajustado, o que é trabalho do Nível 2).